# World Bank Data Foundations
### From raw World Bank Open Data to a cleaned dataset, visual insight, and a first forecast

**Author:** Sunil Mogadati

This notebook is the **hands-on data spine** of the AI Engineer curriculum. You will pull real
data straight from **World Bank Open Data**, clean it the way it actually arrives (messy, with
gaps), engineer features, visualize what matters, and build your **first time-series forecast** —
all on one dataset you can carry into the capstone.

> **Curriculum mapping**
> - **Week 1** — Python for AI, data handling & preprocessing, exploratory data analysis (EDA) and visualization *(most of this notebook)*
> - **Week 3** — time-series forecasting basics *(the final section)*
> - **Week 4** — this same World Bank dataset is your **capstone** foundation

**How to run:** open in Google Colab (or any Jupyter environment) and run the cells top to bottom.
The first cell installs everything you need.


## What you will be able to do by the end
- Pull any indicator for any country from World Bank Open Data using `wbgapi`
- Reshape "wide" data into tidy "long" data (a skill you will use forever)
- Diagnose and handle **missing values** honestly (World Bank data is full of them)
- Engineer features that carry more signal than the raw numbers
- Build four standard visualizations and read them like an analyst
- Split a time series **correctly** and beat a naive baseline with a first forecast

**The mindset for this notebook:** you are not memorizing functions. You are learning the
*decisions* a data-facing AI engineer makes — what to trust, what to drop, what to build.


## 0. Setup

One cell. On a fresh machine (like Colab) it installs the World Bank client, then imports the
four libraries we use throughout.


In [ ]:
# --- Setup: on a fresh machine (e.g. Google Colab) this installs the World Bank client ---
# wbgapi = the "World Bank Group API" python client. It is our single door into
# World Bank Open Data (thousands of development indicators for ~200 economies).
!pip install wbgapi --quiet

import wbgapi as wb          # WHY: pythonic client that hides the raw HTTP calls to the World Bank
import pandas as pd          # WHY: DataFrames = the spreadsheet-in-code we do all data work in
import numpy as np           # WHY: fast numeric arrays + a clean way to represent "missing" (NaN)
import matplotlib.pyplot as plt   # WHY: our plotting library for visualization

pd.set_option("display.max_columns", 20)   # so wide tables are not truncated while teaching
print("Setup complete. wbgapi version:", wb.__version__)


> **✅ You should see:** `Setup complete. wbgapi version: 1.x.x`. If `pip` printed a lot of
> lines above it, that is normal on the first run in Colab.


## 1. Hello, World Bank

The golden rule of learning any data source: **make the smallest possible request work first,**
then grow. Before pulling five indicators for six countries, we pull **one number**.


In [ ]:
# HELLO, WORLD BANK -- the simplest possible request: one indicator, one country, a few years.
#   'NY.GDP.MKTP.CD' = Gross Domestic Product (GDP) in current US dollars
#   'USA'            = United States
hello = wb.data.DataFrame("NY.GDP.MKTP.CD", "USA", time=range(2018, 2023))
hello


> **✅ You should see:** a tiny table with one row (`USA`) and one column per year
> (labelled like `YR2018` ... `YR2022`), with values around `2.0e+13` — that is ~20 **trillion**
> dollars. If you got an error mentioning the network, re-run — the World Bank API occasionally
> times out.

**Layman analogy:** think of `wb.data.DataFrame(...)` as a vending machine. You punch in a code
(the indicator), pick a country and years, and it hands you back a small table. Everything else in
this notebook is just punching in bigger orders.


## 2. Browsing the catalog

World Bank Open Data has **thousands** of indicators. You do not memorize codes — you *search the
catalog*, exactly like searching a library before checking out books.


In [ ]:
# Search the catalog for indicators whose name mentions "life expectancy":
wb.series.info(q="life expectancy")


In [ ]:
# The list of "economies" mixes real COUNTRIES with AGGREGATES (e.g. WLD = World,
# EUU = European Union). WHY this matters: if you accidentally treat an aggregate like a
# country, every average you compute is silently wrong. We hand-pick real countries below.
wb.economy.info()


> **✅ You should see:** two catalog listings. In the first, note the code
> `SP.DYN.LE00.IN` next to "Life expectancy at birth, total (years)". In the second, notice rows
> like `WLD` (World) sitting right next to real countries — that is the trap we just avoided.


## 3. Pulling a real dataset (the tidy reshape)

Now the real thing: a **panel** = several indicators × several countries × many years. We fetch
each indicator, reshape it from **wide** (years spread across columns) to **long/tidy** (one row
per country-year), and merge. This wide→long reshape is one of the most-used moves in all of data
work, so we do it explicitly rather than by magic.


In [ ]:
# Our indicators. WHY these: they are the staples of World Bank development analysis.
indicators = {
    "NY.GDP.MKTP.CD": "gdp_usd",          # GDP, current US$
    "SP.POP.TOTL":    "population",        # total population
    "SP.DYN.LE00.IN": "life_expectancy",   # life expectancy at birth (years)
    "SI.POV.DDAY":    "poverty_rate",      # poverty headcount at $2.15/day (% of pop)  -- SPARSE
    "SE.ADT.LITR.ZS": "literacy_rate",     # adult literacy rate (%)                    -- SPARSE
}
countries = ["USA", "IND", "CHN", "BRA", "NGA", "DEU"]   # deliberately diverse economies
years = range(2000, 2023)

# WHY a loop + melt + merge (instead of one clever call): it is bullet-proof no matter how the
# client orients its output, AND it teaches the wide->long reshape. For each indicator we fetch a
# wide table (countries x years), melt it to (country, year, value), then merge on country+year.
panel = None
for code_, name in indicators.items():
    wide = wb.data.DataFrame(code_, countries, time=years)   # rows = countries, cols = years
    wide = wide.reset_index()                                # move country out of the index
    wide = wide.rename(columns={wide.columns[0]: "country"})
    long = wide.melt(id_vars="country", var_name="year", value_name=name)  # WIDE -> LONG
    panel = long if panel is None else panel.merge(long, on=["country", "year"], how="outer")

# The client labels years like "YR2000"; strip to a real integer (a classic cleaning step).
panel["year"] = panel["year"].astype(str).str.replace("YR", "", regex=False).astype(int)
panel = panel.sort_values(["country", "year"]).reset_index(drop=True)
panel.head(12)


> **✅ You should see:** a tidy table with columns `country`, `year`, and one column per
> indicator. Each row is a single **country-year**. Notice some `poverty_rate` and
> `literacy_rate` cells are already `NaN` — that is real, and we deal with it next.


## 4. First look — interrogate the dataset

A diagnostician asks the same questions of *every* new dataset before touching a model:
how big is it, what types, what ranges, what is missing?


In [ ]:
# Shape, types, and non-null counts in one glance.
print("Shape (rows, columns):", panel.shape)
print()
panel.info()


In [ ]:
# Numeric summary. WHY: min / max / mean instantly reveal scale gaps and absurd values.
panel.describe()


> **✅ You should see:** ~138 rows (6 countries × 23 years), and in `describe()` a **huge**
> spread — `population` ranges from tens of millions to over a billion, `gdp_usd` up to ~1e13.
> Those enormous scale differences are a hint we will need a log transform later.


## 5. Missing data — the real-world core skill

This is the single most important habit with World Bank data. Countries do not run a poverty
survey or a census every year, so gaps are **the norm, not an error**. The engineer's job is to
handle each gap *based on the story behind it*, not with one blind rule.


In [ ]:
# What is missing, and how badly?
missing = panel.isna().sum().sort_values(ascending=False)
missing_pct = (panel.isna().mean() * 100).round(1)
pd.DataFrame({"missing_count": missing, "missing_pct": missing_pct})


In [ ]:
# Strategy depends on the COLUMN and the STORY behind the gap:
#  - poverty_rate / literacy_rate: genuinely sparse (occasional surveys). We KEEP them as NaN and
#    simply avoid modeling on them here -- inventing data we do not have is worse than admitting it.
#  - gdp / population / life_expectancy: nearly complete; small internal gaps can be reasonably
#    forward/back-filled BECAUSE these change slowly year to year.
# WHY groupby("country"): we must fill WITHIN a single country's timeline, never across countries.
for col in ["gdp_usd", "population", "life_expectancy"]:
    panel[col] = panel.groupby("country")[col].ffill().bfill()

print("Remaining missing in core columns:")
print(panel[["gdp_usd", "population", "life_expectancy"]].isna().sum())


### 🔧 Break the System
What if we had "fixed" missing values the lazy way with `panel.fillna(0)`?

- A missing `literacy_rate` would become **0% literacy** — a catastrophic, invented fact.
- A missing `gdp_usd` would become **$0 economy**.

Any model trained on that learns lies. **Lesson:** `fillna(0)` is almost never right for
real-world measurements. Choose a strategy that matches what the number *means*.

> **✅ You should see:** `0` remaining missing for the three core columns after the grouped fill.


## 6. Taming scale — the log transform

`gdp_usd` and `population` span many orders of magnitude (Nigeria vs. the USA). Raw values make
plots unreadable and hurt many models. A **log transform** compresses that range.


In [ ]:
# WHY np.log1p (log of 1+x) instead of np.log: it is safe even if a value is ever exactly 0.
panel["log_gdp"] = np.log1p(panel["gdp_usd"])
panel["log_population"] = np.log1p(panel["population"])
panel[["gdp_usd", "log_gdp", "population", "log_population"]].describe()


> **✅ You should see:** `log_gdp` values in a tidy ~24-32 range instead of up to 1e13. Same
> information, far friendlier scale.


## 7. Feature engineering

**Feature engineering** = building columns that carry *more signal* than the raw inputs. Two
classics on this dataset:


In [ ]:
# 1) GDP per capita: total output divided by people -> a fairer cross-country comparison.
panel["gdp_per_capita"] = panel["gdp_usd"] / panel["population"]

# 2) Year-over-year GDP growth (%), computed WITHIN each country (WHY groupby again: a country's
#    growth compares it to ITS OWN previous year, not to another country).
panel["gdp_growth_pct"] = panel.groupby("country")["gdp_usd"].pct_change() * 100

panel[["country", "year", "gdp_usd", "population", "gdp_per_capita", "gdp_growth_pct"]].head(8)


> **✅ You should see:** a `gdp_per_capita` in the thousands-to-tens-of-thousands range, and a
> `gdp_growth_pct` that is `NaN` for each country's first year (there is no prior year to compare
> to — expected, not a bug).


## 8. Exploratory Data Analysis (EDA) & visualization

Four standard views. For each, we say **what to look for** — a chart you cannot read is decoration.


In [ ]:
# VISUAL 1: GDP per capita over time, one line per country.
fig, ax = plt.subplots(figsize=(9, 5))
for c, grp in panel.groupby("country"):
    ax.plot(grp["year"], grp["gdp_per_capita"], marker="o", markersize=2, label=c)
ax.set_title("GDP per capita over time")
ax.set_xlabel("Year"); ax.set_ylabel("GDP per capita (US$)")
ax.legend()
plt.show()


In [ ]:
# VISUAL 2: most-recent life expectancy, by country.
latest = panel.sort_values("year").groupby("country").tail(1)
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(latest["country"], latest["life_expectancy"])
ax.set_title("Life expectancy (most recent year)")
ax.set_ylabel("Years")
plt.show()


In [ ]:
# VISUAL 3: the classic "Preston curve" -- wealth vs. health.
# Each point is one (country, year). WHY a log x-axis: income is highly skewed.
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(panel["gdp_per_capita"], panel["life_expectancy"], s=10, alpha=0.5)
ax.set_xscale("log")
ax.set_title("Wealth vs. Health (Preston curve)")
ax.set_xlabel("GDP per capita (US$, log scale)"); ax.set_ylabel("Life expectancy (years)")
plt.show()


In [ ]:
# VISUAL 4: correlation between numeric features. WHY: a fast read on what moves together.
num = panel[["gdp_usd", "population", "life_expectancy", "gdp_per_capita", "gdp_growth_pct"]].dropna()
corr = num.corr()
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticks(range(len(corr))); ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, round(corr.iloc[i, j], 2), ha="center", va="center", fontsize=8)
fig.colorbar(im, fraction=0.046, pad=0.04)
ax.set_title("Correlation heatmap")
plt.show()


> **✅ You should see (and how to read it):**
> - **Visual 1** — steeply rising lines for China and India; flatter, higher lines for the USA and Germany.
> - **Visual 3** — a curve that climbs fast at low income then flattens: extra dollars buy huge health gains for poor countries, less for rich ones. This is a *real economic law*, and you just reproduced it from raw data.
> - **Visual 4** — strong positive correlation between `gdp_per_capita` and `life_expectancy`.

### 💼 Explain Like I'm CEO
> "Across these six economies, wealth and health rise together — but with sharply diminishing
> returns. The biggest life-expectancy gains per dollar are in the poorest countries, which is
> exactly where development investment has the most human impact."

That one sentence — not the code — is the deliverable a stakeholder actually wants.


## 9. Week-3 bridge — your first time-series forecast

A **time series** is one value measured repeatedly over time. Forecasting it has one rule that
trips up almost every beginner, and we build the whole section around getting that rule right.

We isolate **one country + one indicator** to get a clean univariate series.


In [ ]:
# Build a clean univariate series: USA GDP over time.
ts = panel[panel["country"] == "USA"][["year", "gdp_usd"]].dropna().sort_values("year").reset_index(drop=True)
ts.head()


In [ ]:
# Plot the series + a 3-year rolling average (smoothing reveals the trend under the wobble).
ts["rolling_3yr"] = ts["gdp_usd"].rolling(window=3).mean()
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(ts["year"], ts["gdp_usd"], marker="o", label="GDP (actual)")
ax.plot(ts["year"], ts["rolling_3yr"], linestyle="--", label="3-year rolling mean")
ax.set_title("USA GDP over time"); ax.set_xlabel("Year"); ax.set_ylabel("GDP (US$)")
ax.legend(); plt.show()


### The one rule: split by TIME, never randomly

In normal machine learning we shuffle rows before splitting into train/test. **For time series
that is forbidden.** If you shuffle, some *future* years leak into training, the model effectively
"sees the answer," and your test score looks amazing — until it meets real future data and
collapses. This is called **data leakage**. We train on the past and test on the future.


In [ ]:
# Split CHRONOLOGICALLY: train on <= 2016, test on the later years. No shuffling.
split_year = 2016
train = ts[ts["year"] <= split_year]
test = ts[ts["year"] > split_year]
print("Train years:", train["year"].min(), "->", train["year"].max(), " | rows:", len(train))
print("Test  years:", test["year"].min(), "->", test["year"].max(), " | rows:", len(test))


In [ ]:
# Two BASELINES. WHY baselines first: if a fancy model cannot beat "carry the last value
# forward", it is not really learning. Baselines are the bar every real model must clear.

# Baseline A -- Naive: predict every future year = the last known (training) value.
last_value = train["gdp_usd"].iloc[-1]
pred_naive = np.full(len(test), last_value)

# Baseline B -- Linear trend: fit a straight line to (year -> GDP) on train, extend it to test.
coeffs = np.polyfit(train["year"], train["gdp_usd"], deg=1)   # returns [slope, intercept]
pred_trend = np.polyval(coeffs, test["year"])

# Evaluate with MAE and RMSE (lower = better). These are the same metrics as
# sklearn.metrics.mean_absolute_error / mean_squared_error -- shown by hand so they are not magic.
def mae(y, yhat):  return np.mean(np.abs(y - yhat))
def rmse(y, yhat): return np.sqrt(np.mean((y - yhat) ** 2))

y_true = test["gdp_usd"].values
print("Naive  -> MAE: {:.3e}   RMSE: {:.3e}".format(mae(y_true, pred_naive), rmse(y_true, pred_naive)))
print("Trend  -> MAE: {:.3e}   RMSE: {:.3e}".format(mae(y_true, pred_trend), rmse(y_true, pred_trend)))


In [ ]:
# See it: both forecasts against the actual test-year values.
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(train["year"], train["gdp_usd"], marker="o", label="Train (actual)")
ax.plot(test["year"], y_true, marker="o", color="black", label="Test (actual)")
ax.plot(test["year"], pred_naive, linestyle="--", label="Naive forecast")
ax.plot(test["year"], pred_trend, linestyle="--", label="Linear-trend forecast")
ax.axvline(split_year, color="gray", linestyle=":", alpha=0.7)
ax.set_title("Forecasts vs. reality"); ax.set_xlabel("Year"); ax.set_ylabel("GDP (US$)")
ax.legend(); plt.show()


> **✅ You should see:** the **linear-trend** forecast has a lower MAE/RMSE than the **naive**
> one, and in the chart it tracks the black "actual" line much more closely. You just beat a
> baseline — the core loop of all forecasting.

### 🔧 Break the System
Try changing the split to a **random** train/test split (e.g. `ts.sample(frac=0.7)` for train).
The error will look *smaller* — and it is a lie. The model borrowed future points it will never
have in production. **Lesson:** with time, always split by the clock.

**Where this goes next (Week 3+):** the naive and linear-trend baselines are step one. Real
forecasting adds seasonality and autocorrelation with models like **ARIMA** or **Prophet** — but
every one of them still has to beat these baselines, and still obeys the split-by-time rule.


## Linear Regression on this data — applying the ML Study 01 math

We derived linear regression by hand (hypothesis → cost function → gradient descent → R²).
Here it is on **real World Bank data**: can a country's **income** predict its **life expectancy**?

- **target (y):** life expectancy at birth
- **feature (x):** `ln(GDP per capita)` — log, because income is highly skewed, which straightens the relationship

`scikit-learn` does the fitting for us; the `theta0` (intercept) and `theta1` (slope) it finds are exactly
the intercept and slope from the lesson.

In [ ]:
# Applying ML Study 01: fit a real linear regression on World Bank data.
# life_expectancy = theta0 + theta1 * ln(GDP per capita)
from sklearn.linear_model import LinearRegression

YEAR = 2021
def _col(x):
    return x.iloc[:, 0] if getattr(x, "ndim", 1) > 1 else x   # single-year pull -> Series

le = wb.data.DataFrame("SP.DYN.LE00.IN", time=YEAR, skipAggs=True, labels=False)   # life expectancy
gp = wb.data.DataFrame("NY.GDP.PCAP.CD", time=YEAR, skipAggs=True, labels=False)   # GDP per capita
reg = pd.DataFrame({"life_exp": _col(le), "gdp_pc": _col(gp)}).dropna()
reg = reg[reg["gdp_pc"] > 0].copy()
reg["log_gdp"] = np.log(reg["gdp_pc"])     # WHY log: income is skewed; log makes it ~linear

X = reg[["log_gdp"]].values                # feature (must be 2D for sklearn)
y = reg["life_exp"].values                 # target

model = LinearRegression().fit(X, y)       # sklearn runs the fit for us (the math from the lesson)
print(f"Countries used: {len(reg)}")
print(f"theta0 (intercept):        {model.intercept_:.2f}")
print(f"theta1 (slope on log GDP): {model.coef_[0]:.2f}")
print(f"R^2:                       {model.score(X, y):.3f}")

# Plot the data + the best-fit line
xs = np.linspace(reg["log_gdp"].min(), reg["log_gdp"].max(), 100)
plt.figure(figsize=(8, 5))
plt.scatter(reg["log_gdp"], y, s=22, alpha=0.6, label="each dot = one country")
plt.plot(xs, model.intercept_ + model.coef_[0] * xs, color="red", lw=2, label="best-fit line")
plt.xlabel("ln(GDP per capita)"); plt.ylabel("Life expectancy (years)")
plt.title("Linear regression: life expectancy vs income (World Bank)")
plt.legend(); plt.show()

> **✅ You should see:** ~200 countries, an intercept **theta0 ≈ 32**, a slope **theta1 ≈ 4.4**, and
> **R² ≈ 0.71** — income alone explains ~71% of the differences in life expectancy across countries.
> The scatter shows richer countries living longer, with the red best-fit line running through them.
>
> **Interpretation (interview gold):** theta1 ≈ 4.4 means each unit of `ln(GDP per capita)` adds ~4.4 years
> — i.e. **every doubling of income buys ≈ 3 more years of life** (4.4 × ln 2). The remaining 29% is what
> other features (healthcare, education, conflict) would capture in a **multiple regression**.
>
> **Your turn:** swap in a different indicator, or bring your own dataset, and repeat these same few lines
> — that becomes the project on your resume.

## 10. Architect's Decision Checklist

The judgment layer — what a senior engineer actually decides on a job like this:

| Decision | What we chose here | Why |
|---|---|---|
| Data source | World Bank Open Data via `wbgapi` | Free, authoritative, reproducible; no license friction |
| Data shape | Reshape wide → tidy long | One row per country-year makes every later step simpler |
| Missing values | Keep sparse survey columns as NaN; grouped ffill/bfill for slow-moving core columns | Match the strategy to what the number *means*; never invent data |
| Scale | Log transform GDP & population | Skewed, multi-order-of-magnitude values break plots and many models |
| Features | GDP per capita, YoY growth | Ratios and changes carry more signal than raw totals |
| Time-series split | Chronological, never random | Random splitting leaks the future → fake-good scores |
| First model | Naive + linear-trend baselines | If a complex model cannot beat these, it is not learning |

**Carry this forward:** for your capstone you will swap in a different indicator or country,
add a real model (Week 2's scikit-learn / deep learning), and serve it as an API (Week 3's
FastAPI lab) — but these seven decisions stay the same.


## 11. Wrap-up & where this connects

You took raw World Bank data and produced: a **cleaned tidy dataset**, engineered **features**,
four **visualizations** with plain-English readings, and a **first forecast that beats a baseline**.

**This notebook is your capstone spine.** In Week 4 you will:
1. Pick your own indicator(s) / countries from the same catalog
2. Preprocess and visualize (Sections 3–8 here)
3. Train and compare real models (Week 2 material)
4. Deploy the chosen model as an API or dashboard (Week 3 FastAPI lab)
5. Document and present — with an "Explain Like I'm CEO" summary like the one above

**Next-step learning:** scikit-learn model comparison, ARIMA/Prophet for seasonality,
model serving and drift monitoring.


## Glossary (terms expanded on first use)

| Term | Meaning |
|---|---|
| **World Bank Open Data** | Free public database of development indicators for ~200 economies |
| **`wbgapi`** | World Bank Group API — the python client we use to fetch that data |
| **Indicator** | A single measured series (e.g. `NY.GDP.MKTP.CD` = GDP in current US$) |
| **Economy** | The World Bank's term for a country *or* an aggregate (e.g. `WLD` = World) |
| **Panel data** | Multiple entities (countries) measured over multiple time periods |
| **Wide vs. long (tidy)** | Wide = years spread across columns; long/tidy = one row per country-year |
| **NaN** | "Not a Number" — how pandas marks a missing value |
| **Feature engineering** | Building new columns that carry more signal than the raw inputs |
| **EDA** | Exploratory Data Analysis — inspecting/visualizing data before modeling |
| **Log transform** | Replacing x with log(x) to compress a skewed, wide-ranging scale |
| **Time series** | One value measured repeatedly over time |
| **Data leakage** | Letting information from the future (or the test set) into training |
| **Baseline** | A deliberately simple prediction every real model must beat |
| **MAE / RMSE** | Mean Absolute Error / Root Mean Squared Error — lower is better |


---
*World Bank Data Foundations — Author: Sunil Mogadati.*
*Built on public World Bank Open Data (`data.worldbank.org`). Reuse and adapt for your own capstone.*
